# Cart-to-Purchase Signal EDA

**Owner:** one person (dedicated cart→purchase signal analysis)

This notebook builds on Step 1 (`DataIngestion_Subsample.ipynb`) and asks the question that most directly motivates the project hypothesis:

> Does add-to-cart behavior carry useful information about subsequent purchase of the same item — and how soon after carting does that purchase tend to happen?

We answer that with two analyses on the **active-user subsampled cohort**:

1. **Conversion rate** from add-to-cart to subsequent purchase (user–SKU pairs)
2. **Time-gap** between first cart-add and first later purchase for converted pairs

**Hypothesis link.** If cart→purchase conversion is material (not near-zero) and many conversions happen within a short window, then cart history is a more *recent* intent signal than sparse purchase history alone — supporting cart-augmented sequential recommenders over purchase-only baselines.

> **How to run:** After `DataIngestion_Subsample.ipynb` writes the active cohort, place (or symlink) outputs under `data/subsampled/` and *Restart & Run All*. Numbers and figures are produced at execution time from that cohort.


In [ ]:
"""Cart-to-purchase signal EDA on the active-user subsampled cohort."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
})


## 0. Load subsampled cohort

Prefer `data/subsampled/` (repo-relative output of Step 1). Falls back to the local Desktop path used in the ingestion notebook.


In [ ]:
CANDIDATE_DIRS = [
    Path("data/subsampled"),
    Path("Desktop/DS5500/data/subsampled"),
    Path.home() / "Desktop" / "DS5500" / "data" / "subsampled",
]

DATA_DIR = next((p for p in CANDIDATE_DIRS if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find subsampled data. Run DataIngestion_Subsample.ipynb first, "
        f"or place parquet files under one of: {[str(p) for p in CANDIDATE_DIRS]}"
    )

FILES = {
    "add_to_cart": "add_to_cart_subsampled.parquet",
    "product_buy": "product_buy_subsampled.parquet",
    "remove_from_cart": "remove_from_cart_subsampled.parquet",
}

print(f"Loading from: {DATA_DIR.resolve()}")

events = {}
for name, fname in FILES.items():
    path = DATA_DIR / fname
    df = pd.read_parquet(path)
    df = df.drop_duplicates().copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    n_null_ts = df["timestamp"].isna().sum()
    if n_null_ts:
        df = df.dropna(subset=["timestamp"])
    events[name] = df
    print(
        f"{name:18s}: {len(df):>10,} rows | "
        f"{df['client_id'].nunique():>8,} users | "
        f"{df['sku'].nunique():>8,} SKUs | "
        f"null timestamps dropped={n_null_ts}"
    )

add_to_cart = events["add_to_cart"]
product_buy = events["product_buy"]
remove_from_cart = events["remove_from_cart"]

print(
    f"\nObservation window: "
    f"{min(df['timestamp'].min() for df in events.values())} → "
    f"{max(df['timestamp'].max() for df in events.values())}"
)


## 1. Build user–SKU cart and purchase anchors

We analyze **unique (client_id, sku) pairs**. For each pair we take the earliest cart-add and the earliest purchase. A pair is a **conversion** if a purchase of that SKU occurs **strictly after** the first cart-add for the same user.

This matches the preliminary EDA definition and avoids counting purchases that preceded the cart signal.


In [ ]:
first_cart = (
    add_to_cart.groupby(["client_id", "sku"], as_index=False)["timestamp"]
    .min()
    .rename(columns={"timestamp": "first_cart_time"})
)

first_purchase = (
    product_buy.groupby(["client_id", "sku"], as_index=False)["timestamp"]
    .min()
    .rename(columns={"timestamp": "first_purchase_time"})
)

cart_purchase = first_cart.merge(
    first_purchase, on=["client_id", "sku"], how="left"
)

cart_purchase["subsequently_purchased"] = (
    cart_purchase["first_purchase_time"].notna()
    & (cart_purchase["first_purchase_time"] > cart_purchase["first_cart_time"])
)

cart_purchase["time_to_purchase"] = (
    cart_purchase["first_purchase_time"] - cart_purchase["first_cart_time"]
)

print(f"Unique carted user–SKU pairs : {len(cart_purchase):,}")
print(f"Unique purchased user–SKU pairs: {len(first_purchase):,}")
cart_purchase.head()


## 2. Conversion rate: add-to-cart → purchase

### 2a. Overall pair-level conversion


In [ ]:
n_cart_pairs = len(cart_purchase)
n_converted = int(cart_purchase["subsequently_purchased"].sum())
conversion_rate = cart_purchase["subsequently_purchased"].mean()

# Purchases that had a prior cart vs purchases with no prior cart of that SKU
purchases_with_prior_cart = first_purchase.merge(
    first_cart, on=["client_id", "sku"], how="left"
)
purchases_with_prior_cart["had_prior_cart"] = (
    purchases_with_prior_cart["first_cart_time"].notna()
    & (
        purchases_with_prior_cart["first_cart_time"]
        < purchases_with_prior_cart["first_purchase_time"]
    )
)
prior_cart_share = purchases_with_prior_cart["had_prior_cart"].mean()

# User-level: among users with ≥1 cart event, what share convert at least once?
users_with_cart = set(first_cart["client_id"])
users_with_conversion = set(
    cart_purchase.loc[cart_purchase["subsequently_purchased"], "client_id"]
)
user_conversion_rate = len(users_with_conversion) / len(users_with_cart)

conversion_summary = pd.DataFrame(
    {
        "Metric": [
            "Carted user–SKU pairs",
            "Subsequently purchased pairs",
            "Pair-level cart→purchase conversion rate",
            "Purchased pairs with a prior cart of same SKU",
            "Users with ≥1 cart event",
            "Users with ≥1 cart→purchase conversion",
            "User-level conversion rate (among cart users)",
        ],
        "Result": [
            f"{n_cart_pairs:,}",
            f"{n_converted:,}",
            f"{conversion_rate * 100:.2f}%",
            f"{prior_cart_share * 100:.2f}%",
            f"{len(users_with_cart):,}",
            f"{len(users_with_conversion):,}",
            f"{user_conversion_rate * 100:.2f}%",
        ],
    }
)

print(
    f"Pair-level conversion : {n_converted:,} / {n_cart_pairs:,} "
    f"= {conversion_rate * 100:.2f}%"
)
print(
    f"Share of purchases preceded by same-SKU cart: {prior_cart_share * 100:.2f}%"
)
print(
    f"User-level conversion : {len(users_with_conversion):,} / "
    f"{len(users_with_cart):,} = {user_conversion_rate * 100:.2f}%"
)
conversion_summary


### 2b. Remove-from-cart as a competing signal

If a user removes the SKU from cart before purchasing, that may mark abandoned intent. We flag pairs where the earliest remove happens after the first cart and before any subsequent purchase.


In [ ]:
first_remove = (
    remove_from_cart.groupby(["client_id", "sku"], as_index=False)["timestamp"]
    .min()
    .rename(columns={"timestamp": "first_remove_time"})
)

cart_purchase_rm = cart_purchase.merge(
    first_remove, on=["client_id", "sku"], how="left"
)

has_remove = cart_purchase_rm["first_remove_time"].notna()
remove_after_cart = (
    has_remove
    & (cart_purchase_rm["first_remove_time"] > cart_purchase_rm["first_cart_time"])
)
# Intervening remove: after cart, and either before purchase or no purchase at all
cart_purchase_rm["removed_before_purchase"] = remove_after_cart & (
    ~cart_purchase_rm["subsequently_purchased"]
    | (
        cart_purchase_rm["first_remove_time"]
        < cart_purchase_rm["first_purchase_time"]
    )
)

remove_breakdown = (
    cart_purchase_rm.groupby("removed_before_purchase")["subsequently_purchased"]
    .agg(n_pairs="size", n_converted="sum", conversion_rate="mean")
    .reset_index()
)
remove_breakdown["removed_before_purchase"] = remove_breakdown[
    "removed_before_purchase"
].map({False: "No intervening remove", True: "Removed before purchase / no buy"})
remove_breakdown["conversion_rate"] = (
    remove_breakdown["conversion_rate"] * 100
).round(2)

print("Conversion by whether an intervening remove-from-cart occurred:")
remove_breakdown


### 2c. Visualize conversion composition


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Left: converted vs not among carted pairs
labels_left = ["Converted\n(cart → buy)", "Not converted"]
values_left = [conversion_rate * 100, (1 - conversion_rate) * 100]
bars_left = axes[0].bar(labels_left, values_left, color=["#2a6f97", "#adb5bd"])
axes[0].set_ylabel("Percentage of carted user–SKU pairs")
axes[0].set_title("Cart → Purchase Conversion")
axes[0].set_ylim(0, 100)
for bar, value in zip(bars_left, values_left):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.5,
        f"{value:.1f}%",
        ha="center",
    )

# Right: purchases with vs without prior cart
labels_right = ["Prior same-SKU cart", "No prior cart"]
values_right = [prior_cart_share * 100, (1 - prior_cart_share) * 100]
bars_right = axes[1].bar(labels_right, values_right, color=["#2a6f97", "#adb5bd"])
axes[1].set_ylabel("Percentage of purchased user–SKU pairs")
axes[1].set_title("Purchases Preceded by Cart Signal")
axes[1].set_ylim(0, 100)
for bar, value in zip(bars_right, values_right):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.5,
        f"{value:.1f}%",
        ha="center",
    )

fig.tight_layout()
fig.savefig("cart_to_purchase_conversion.png", dpi=200, bbox_inches="tight")
plt.show()


## 3. Time-gap analysis: cart-add → purchase

Restrict to converted pairs and measure how long after the first cart-add the purchase occurs.


In [ ]:
converted = cart_purchase.loc[
    cart_purchase["subsequently_purchased"]
].copy()

converted["gap_hours"] = converted["time_to_purchase"].dt.total_seconds() / 3600.0
converted["gap_days"] = converted["gap_hours"] / 24.0

# Guard against any non-positive gaps (should already be filtered)
converted = converted[converted["gap_hours"] > 0].copy()

gap_desc = converted["gap_hours"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

print(f"Converted pairs with positive gap: {len(converted):,}")
print("\nTime-to-purchase (hours):")
print(gap_desc)

gap_table = pd.DataFrame(
    {
        "Statistic": [
            "Median gap (hours)",
            "Median gap (days)",
            "P25 (hours)",
            "P75 (hours)",
            "P90 (hours)",
            "Mean gap (hours)",
        ],
        "Value": [
            f"{converted['gap_hours'].median():.2f}",
            f"{converted['gap_days'].median():.2f}",
            f"{converted['gap_hours'].quantile(0.25):.2f}",
            f"{converted['gap_hours'].quantile(0.75):.2f}",
            f"{converted['gap_hours'].quantile(0.90):.2f}",
            f"{converted['gap_hours'].mean():.2f}",
        ],
    }
)
gap_table


### 3a. Cumulative conversion within time windows

What fraction of carted pairs convert within 1 hour, 24 hours, 7 days, etc.? This frames how "fresh" the cart signal is for next-purchase ranking.


In [ ]:
windows = [
    ("≤ 1 hour", 1),
    ("≤ 6 hours", 6),
    ("≤ 24 hours", 24),
    ("≤ 3 days", 24 * 3),
    ("≤ 7 days", 24 * 7),
    ("≤ 14 days", 24 * 14),
    ("≤ 30 days", 24 * 30),
    ("Ever (any later purchase)", np.inf),
]

rows = []
for label, hours in windows:
    if np.isinf(hours):
        n = n_converted
    else:
        n = int((converted["gap_hours"] <= hours).sum())
    rows.append(
        {
            "Window": label,
            "Converted pairs": n,
            "% of carted pairs": round(100 * n / n_cart_pairs, 2),
            "% of eventual converters": round(100 * n / max(n_converted, 1), 2),
        }
    )

window_table = pd.DataFrame(rows)
print("Cumulative cart→purchase conversion by elapsed time:")
window_table


### 3b. Time-gap distributions


In [ ]:
# Cap extreme tails for readability in the histogram (keep full data for CDF)
hist_cap_hours = converted["gap_hours"].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Histogram of gaps (hours), log-friendly bins on capped sample
capped = converted.loc[converted["gap_hours"] <= hist_cap_hours, "gap_hours"]
axes[0].hist(capped, bins=50, color="#2a6f97", edgecolor="white", linewidth=0.4)
axes[0].axvline(
    converted["gap_hours"].median(),
    color="#bc4749",
    linestyle="--",
    linewidth=1.5,
    label=f"Median = {converted['gap_hours'].median():.1f}h",
)
axes[0].set_xlabel("Hours from first cart-add to purchase")
axes[0].set_ylabel("Converted user–SKU pairs")
axes[0].set_title("Time-Gap Distribution (capped at P99)")
axes[0].legend()

# Empirical CDF of gap hours (log x-scale)
sorted_gaps = np.sort(converted["gap_hours"].values)
cdf = np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps)
axes[1].plot(sorted_gaps, cdf, color="#2a6f97", linewidth=2)
axes[1].set_xscale("log")
axes[1].set_xlabel("Hours from first cart-add to purchase (log scale)")
axes[1].set_ylabel("Cumulative share of converters")
axes[1].set_title("Empirical CDF of Cart → Purchase Lag")
axes[1].axhline(0.5, color="#adb5bd", linestyle=":", linewidth=1)
axes[1].axvline(
    converted["gap_hours"].median(),
    color="#bc4749",
    linestyle="--",
    linewidth=1.5,
)

# Mark common windows on the CDF
for label, hours in [("1h", 1), ("24h", 24), ("7d", 24 * 7)]:
    share = float((converted["gap_hours"] <= hours).mean())
    axes[1].scatter([hours], [share], color="#bc4749", zorder=3)
    axes[1].annotate(
        f"{label}\n{share * 100:.0f}%",
        (hours, share),
        textcoords="offset points",
        xytext=(6, 6),
        fontsize=8,
    )

fig.tight_layout()
fig.savefig("cart_to_purchase_time_gap.png", dpi=200, bbox_inches="tight")
plt.show()


### 3c. Same-day / same-session style buckets

Coarse buckets help storytelling in the proposal / report without relying on a session ID (the Synerise release does not provide one).


In [ ]:
def gap_bucket(hours: float) -> str:
    if hours <= 1:
        return "0–1h"
    if hours <= 24:
        return "1–24h"
    if hours <= 24 * 7:
        return "1–7d"
    if hours <= 24 * 30:
        return "7–30d"
    return ">30d"

converted["gap_bucket"] = converted["gap_hours"].map(gap_bucket)
bucket_order = ["0–1h", "1–24h", "1–7d", "7–30d", ">30d"]
bucket_counts = (
    converted["gap_bucket"]
    .value_counts()
    .reindex(bucket_order)
    .fillna(0)
    .astype(int)
)
bucket_pct = (100 * bucket_counts / bucket_counts.sum()).round(2)

bucket_df = pd.DataFrame(
    {
        "Bucket": bucket_order,
        "Converted pairs": bucket_counts.values,
        "% of converters": bucket_pct.values,
    }
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(bucket_df["Bucket"], bucket_df["% of converters"], color="#2a6f97")
ax.set_ylabel("% of converted user–SKU pairs")
ax.set_xlabel("Time from first cart-add to purchase")
ax.set_title("Where Conversions Fall in Time")
ax.set_ylim(0, max(bucket_df["% of converters"].max() * 1.15, 10))
for bar, value in zip(bars, bucket_df["% of converters"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.8,
        f"{value:.1f}%",
        ha="center",
        fontsize=9,
    )
fig.tight_layout()
fig.savefig("cart_to_purchase_gap_buckets.png", dpi=200, bbox_inches="tight")
plt.show()

bucket_df


## 4. Takeaways for the project hypothesis

Fill in the printed numbers after running on the real subsampled cohort. The qualitative interpretation below is what this EDA is designed to support or challenge.


In [ ]:
# Share of converters within key windows (for the written takeaway)
pct_within_24h = 100 * (converted["gap_hours"] <= 24).mean()
pct_within_7d = 100 * (converted["gap_hours"] <= 24 * 7).mean()
median_hours = converted["gap_hours"].median()

takeaways = f"""
Cart-to-purchase signal — key numbers (active cohort)
------------------------------------------------------
1. Pair-level conversion rate: {conversion_rate * 100:.2f}%
   ({n_converted:,} of {n_cart_pairs:,} carted user–SKU pairs later purchased)

2. Among purchasing user–SKU pairs, {prior_cart_share * 100:.2f}% had a prior
   same-SKU cart-add — cart is often an observable precursor of purchase.

3. Median cart→purchase lag: {median_hours:.2f} hours
   ({pct_within_24h:.1f}% of converters within 24h;
    {pct_within_7d:.1f}% within 7 days)

Interpretation for modeling
---------------------------
- A non-trivial conversion rate means cart history is informative about what
  a user may buy next — not pure noise relative to purchase-only baselines.
- Short time gaps mean cart events are typically *more recent* than older
  purchases in sparse histories, which is exactly the gap our proposal cites.
- Incomplete conversion (most carted items are never bought) also justifies
  keeping remove-from-cart / non-conversion as negative or weak signals rather
  than treating every cart-add as a guaranteed future purchase.
"""
print(takeaways)

final_summary = pd.DataFrame(
    {
        "Metric": [
            "Pair-level cart→purchase conversion",
            "Purchases with prior same-SKU cart",
            "Median cart→purchase lag (hours)",
            "% of converters within 24 hours",
            "% of converters within 7 days",
        ],
        "Result": [
            f"{conversion_rate * 100:.2f}%",
            f"{prior_cart_share * 100:.2f}%",
            f"{median_hours:.2f}",
            f"{pct_within_24h:.1f}%",
            f"{pct_within_7d:.1f}%",
        ],
    }
)
final_summary


## 5. Outputs to share with the team

Saved figures (cwd):

- `cart_to_purchase_conversion.png`
- `cart_to_purchase_time_gap.png`
- `cart_to_purchase_gap_buckets.png`

Next steps after this EDA:

1. Join `product_properties` and stratify conversion / lag by anonymized `category` (feeds Phase 2 category clustering).
2. Use conversion labels and time-aware cart features when constructing multi-behavior sequences for SASRec.
